In [55]:
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge

## Novi error funkciji

In [56]:
import numpy as np
from scipy.interpolate import interp1d

def _prepare_curve_for_interp(curve):
    """
    Ensure curve is (N,3), sorted by x, with duplicate x merged (avg y,z).
    Returns x, y, z arrays suitable for interp1d.
    """
    c = np.asarray(curve, dtype=float)
    if c.ndim != 2 or c.shape[1] != 3:
        raise ValueError("curve must have shape (N, 3)")

    order = np.argsort(c[:, 0])
    c = c[order]

    x = c[:, 0]
    y = c[:, 1]
    z = c[:, 2]

    # merge duplicate x by averaging y,z
    uniq_x, inverse = np.unique(x, return_inverse=True) # vrednosti in indeksi
    if len(uniq_x) < len(x):    #znebi se podvojenih x vrednosti tako da averaga vse ostale vrednosti pri tem x za y in z
        y_acc = np.zeros_like(uniq_x, dtype=float)      # vsota vseh y vrednosti pri enakem x
        z_acc = np.zeros_like(uniq_x, dtype=float)
        count = np.zeros_like(uniq_x, dtype=float)

        np.add.at(y_acc, inverse, y)    #prekopira in sesteje vrednost y v vrstnem redu inverse v array y_acc za se ponovljene x vrednosti
        np.add.at(z_acc, inverse, z)
        np.add.at(count, inverse, 1.0)

        y = y_acc / count       #vsoto delimo s stevilom tock
        z = z_acc / count
        x = uniq_x

    return x, y, z, c  # c is sorted, not necessarily same length if duplicates merged

when writing an article explain why it's ok to take a value at each of the whole numbers (integers) of x.

In [57]:
def error_fun_x_aligned(curveA, curveB):
    """
    Mean 3D distance between two 3D curves, compared at the SAME integer x-values.

    - First point compared directly (even if x differs).
    - Main part: compare at common integer x values (intersection).
    - Tail penalty: remaining integer-x part of the longer curve is compared to
      the LAST point of the shorter curve.

    Returns:
        mean_dist (float): average Euclidean distance (same units as input coords).
    """
    A = np.asarray(curveA, dtype=float)
    B = np.asarray(curveB, dtype=float)

    d0 = np.linalg.norm(A[0] - B[0])

    xA, yA, zA, A_sorted = _prepare_curve_for_interp(A)
    xB, yB, zB, B_sorted = _prepare_curve_for_interp(B)

    fAy = interp1d(xA, yA, kind="linear", bounds_error=False, fill_value="extrapolate")
    fAz = interp1d(xA, zA, kind="linear", bounds_error=False, fill_value="extrapolate")
    fBy = interp1d(xB, yB, kind="linear", bounds_error=False, fill_value="extrapolate")
    fBz = interp1d(xB, zB, kind="linear", bounds_error=False, fill_value="extrapolate")

    xA_min, xA_max = xA.min(), xA.max()
    xB_min, xB_max = xB.min(), xB.max()
    xA_int = np.arange(np.ceil(xA_min), np.floor(xA_max) + 1)
    xB_int = np.arange(np.ceil(xB_min), np.floor(xB_max) + 1)
    x_common = np.intersect1d(xA_int, xB_int)

    start_after = max(A[0, 0], B[0, 0])
    x_common = x_common[x_common > start_after]

    dists = [d0]

    if len(x_common) > 0:
        Ay = fAy(x_common); Az = fAz(x_common)
        By = fBy(x_common); Bz = fBz(x_common)

        A_pts = np.column_stack([x_common, Ay, Az])
        B_pts = np.column_stack([x_common, By, Bz])

        dists.extend(np.linalg.norm(A_pts - B_pts, axis=1))


    endA = xA_max
    endB = xB_max

    if endA == endB:
        return float(np.mean(dists))

    if endA < endB:
        last_short = A_sorted[-1]  #ustavili smo se na zadnji točki A
        x_extra = np.arange(np.floor(endA) + 1, np.floor(endB) + 1)

        if len(x_extra) > 0:
            By = fBy(x_extra); Bz = fBz(x_extra)
            B_tail = np.column_stack([x_extra, By, Bz])
            dists.extend(np.linalg.norm(B_tail - last_short[None, :], axis=1))

        dists.append(np.linalg.norm(B_sorted[-1] - last_short))

    else:
        last_short = B_sorted[-1]
        x_extra = np.arange(np.floor(endB) + 1, np.floor(endA) + 1)

        if len(x_extra) > 0:
            Ay = fAy(x_extra); Az = fAz(x_extra)
            A_tail = np.column_stack([x_extra, Ay, Az])
            dists.extend(np.linalg.norm(A_tail - last_short[None, :], axis=1))

        dists.append(np.linalg.norm(A_sorted[-1] - last_short))

    return float(np.mean(dists))


## stari error funkciji

In [58]:
def interpolate_curve_by_x(curve):
    """
    Interpolates a 3D curve by x-coordinate (samplede at integer values) and returns (x, y, z),
    keeping the original start and end points exactly.

    Args:
        curve : array-like of shape (N, 3)
            Each row is [x, y, z].

    Returns:
        result : ndarray of shape (M, 3)
            Interpolated coordinates at integer x plus original endpoints.
    """

    curve = np.asarray(curve, dtype=float)
    if curve.ndim != 2 or curve.shape[1] != 3:
        raise ValueError("Data has the wrong shape")

    x = curve[:, 0]
    y = curve[:, 1]
    z = curve[:, 2]    

    x_new = np.arange(np.ceil(x.min()) + 0, np.floor(x.max()) + 1)

    #def f_y(x): values = interp1d(x, y, kind="linear", bounds_error=False, fill_value="extrapolate") return values
    #def f_z(x): values = interp1d(x, z, kind="linear", bounds_error=False, fill_value="extrapolate") return values
    
    f_y =  interp1d(x, y, kind="linear", bounds_error=False, fill_value="extrapolate")
    f_z = interp1d(x, z, kind="linear", bounds_error=False, fill_value="extrapolate")

    if len(x_new) > 0 :# or x[-1] != x_new[-1]:
        y_new = f_y(x_new)
        z_new = f_z(x_new)
        result = np.vstack([curve[0], np.column_stack((x_new, y_new, z_new)), curve[-1]])
    else:
        result = curve

    return result

In [59]:
def error_fun(curveA, curveB):
    curve1 = interpolate_curve_by_x(curveA)
    curve2 = interpolate_curve_by_x(curveB)

    #curve1 = interpolate_curve_by_arclength(curveA)
    #curve2 = interpolate_curve_by_arclength(curveB)

    if len(curve1) < len(curve2):
        longer = curve2
        shorter = curve1
    else:
        longer = curve1
        shorter = curve2

    longer1 = longer[:len(shorter)]
    shorter1 = shorter

    pad_len = len(longer) - len(shorter)
    if pad_len > 0:
        last_point = shorter[-1][None, :]   #seznam ene zadnje točke
        pad = np.repeat(last_point, pad_len, axis=0)    #ponovimo
        shorter2 = pad
        longer2 = longer[len(shorter):]

        diffs = np.vstack([longer1 - shorter1, longer2 - shorter2])
    else:
        diffs = longer1 - shorter1

    dists = np.linalg.norm(diffs, axis=1)
    return np.mean(dists)

## Plot

In [60]:
import os
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import math

def plot(X_seq, X_sim, j, name):
    """
    Plot actual vs simulated trajectory in 3D with projections.
    Args:
        X_seq (ndarray): ground truth state sequence (T, state_dim)
        X_sim (ndarray): simulated state sequence (T, state_dim)
        j (int): index of jump
        name (str): identifier for saving plots """

    save_dir = f"plots/{name}"
    os.makedirs(save_dir, exist_ok=True)

    
    actual = X_seq[:, :3]   #vzame X,Y,Z iz [x, y, z, vx, vy, vz]
    sim = X_sim[:, :3]          # same iz simulacije

    Xa, Ya, Za = actual[:,0], actual[:,1], actual[:,2]
    Xs, Ys, Zs = sim[:,0], sim[:,1], sim[:,2]

    #napaka = np.mean(np.linalg.norm(actual - sim, axis=1))
    #napaka = error_fun(actual, sim)
    napaka = error_fun_x_aligned(actual, sim)
    length = (Xs[-1]**2 + Ys[-1]**2 + Zs[-1]**2)**(1/2)
    length_a = (Xa[-1]**2 + Ya[-1]**2 + Za[-1]**2)**(1/2)

    fig = plt.figure(figsize=(8, 5))
    ax = fig.add_subplot(111, projection='3d')

    ax.plot3D(Xa, Ya, Za, label="Actual", color="blue")
    ax.plot3D(Xs, Ys, Zs, label="Simulated", color="red", linestyle="--")


    # Ground projections
    ax.plot(Xa, 15, Za, color="blue", alpha=0.3, linestyle=':')
    ax.plot(Xs, 15, Zs, color="red", alpha=0.3, linestyle=':')

    ax.plot(0, Ya, Za, color="blue", alpha=0.3, linestyle=':')
    ax.plot(0, Ys, Zs, color="red", alpha=0.3, linestyle=':')


    ax.set_title(f"Actual vs Simulated Trajectory\n Error: {napaka:.3f}, Simulated length: {length:.1f},  Actual length: {length_a:.1f}")
    ax.set_xlabel("X [m]")
    ax.set_ylabel("Y [m]")
    ax.set_zlabel("Z [m]")
    ax.legend()
    ax.set_box_aspect([1,1,1])
    ax.set_ylim(-15, 15)

    plt.legend()
    #plt.tight_layout()

    filename = f"2SSM flight_simulation{j + 1}.png"
    plt.savefig(os.path.join(save_dir, filename), dpi=300)
    plt.close(fig)
    #plt.show()


In [61]:
#wind_features = ["Speed", "Tangent", "Cross", "Turbulence", "Speed_quad", "Tangent_quad", "Cross_quad", "Turbulence_quad"]
#
wind_features = ["Speed", "Tangent", "Cross", "Turbulence"] 

zones = {
    "takeoff": ["W1", "W2", "W3", "W4"],
    "mid":     ["W5", "W6", "W7", "W8"],
    "landing": ["W9", "W10", "W11", "W12"]}

In [62]:

def preprocess_flight_normalized(df, step=0.05):
    """
    Preprocess a normalized flight dataframe (already interpolated) to generate states (X), observations (Y), and controls (U).
    
    Args:
        df (pd.DataFrame): Normalized flight dataframe (fixed time step).
        step (float): Time step between rows (used for derivatives).
        
    Returns:
        states (np.ndarray): State matrix [time_steps, state_dim].
        observations (np.ndarray): Observation matrix [time_steps, obs_dim].
        controls (np.ndarray): Control matrix [time_steps, control_dim].
    """

    df.columns = df.columns.str.strip()   #izloči imena stolpcev
    df = df.iloc[1:]
    
    df = df.ffill().bfill()   #back fill za manjkajoče vrednosti
    
    x = df["X [m]"].to_numpy()   #save values
    y = df["Y [m]"].to_numpy()
    z = df["Z [m]"].to_numpy()
    
    dt = step
    vx = np.gradient(x, dt)
    vy = np.gradient(y, dt)
    vz = np.gradient(z, dt)

    #speed_hor = df.get("Speed hor. [km/h]", pd.Series([0]*len(x))).to_numpy() * (1000/3600)
    #speed_ver = df.get("Speed vert. [km/h]", pd.Series([0]*len(x))).to_numpy() * (1000/3600)
    speed = df.get("Speed resulting [km/h]", pd.Series([0]*len(x))).to_numpy() * (1000/3600)

    opening = df.get("Opening Angle [°]", pd.Series([0]*len(x))).to_numpy()
    roll_L = df.get("Roll Angle Left [°]", pd.Series([0]*len(x))).to_numpy()
    roll_R = df.get("Roll Angle Right [°]", pd.Series([0]*len(x))).to_numpy()
    yaw_L = df.get("Yaw Angle Left [°]", pd.Series([0]*len(x))).to_numpy()
    yaw_R = df.get("Yaw Angle Right [°]", pd.Series([0]*len(x))).to_numpy()
    stall_L = df.get("Stalling Angle Left [°]", pd.Series([0]*len(x))).to_numpy()
    stall_R = df.get("Stalling Angle Right [°]", pd.Series([0]*len(x))).to_numpy()

    zero = df.get("", pd.Series([0]*len(x))).to_numpy()

    states = np.stack([x, y, z, vx, vy, vz, speed, opening, roll_L, roll_R, yaw_L, yaw_R, stall_L, stall_R], axis=1)   #zgradimo state vector X
    #states = np.stack([x, y, z, vx, vy, vz, speed_hor, speed_ver, speed, opening, roll_L, roll_R, yaw_L, yaw_R, stall_L, stall_R], axis=1)

    height = df.get("Height above ground [m]", pd.Series([0]*len(x))).to_numpy()

    observations = np.stack([x, y, z, height], axis=1)   #zgradimo observation vector Y

    zone_feature_avgs = []
    values = []
    cols = []

    #for feature in wind_features:
    #    for sensor_numb in range(12):
    #        sensor = f"W{sensor_numb + 1}"
    #        cols.append(f"{sensor}_{feature}")
    #        #print(cols)
    #for col in cols:
    #    val = df.get(col, pd.Series([0]*len(x))).to_numpy() 
    #    values.append(val)
    #
    #controls = np.stack(values, axis=1)  # shape: (time_steps, 12)

    #for feature in wind_features:
    #    for zone, sensors in zones.items():
    #        cols = [f"{sensor}_{feature}" for sensor in sensors if f"{sensor}_{feature}" in df.columns]
    #        avg_feature = df[cols].mean(axis=1).to_numpy()
    #        zone_feature_avgs.append(avg_feature)
    #
    #controls = np.stack(zone_feature_avgs, axis=1)  # shape: (time_steps, 12)

    for feature in wind_features:
        for zone, sensors in zones.items():
            cols = [f"{sensor}_{feature}" for sensor in sensors if f"{sensor}_{feature}" in df.columns]
            avg_feature = df[cols].mean(axis=1).to_numpy()
            zone_feature_avgs.append(avg_feature)
            angles = [speed, opening, roll_L, roll_R, yaw_L, yaw_R, stall_L, stall_R]
    for feat in angles:
        zone_feature_avgs.append(feat)

    controls = np.stack(zone_feature_avgs, axis=1)
    
    return states, observations, controls


In [63]:
import os
import pandas as pd
import numpy as np

normalized_folder = r'C:\Users\vsi\Desktop\ijs\smucarski_skoki\project\simulation\2024_03_Planica_12_winds\cleaned\cleaned_quad\normalized_quad'
combined_output = os.path.join(normalized_folder, "combined_dataset.npz")
file_names = []

X_list, Y_list, U_list = [], [], []

for filename in os.listdir(normalized_folder):
    if filename.endswith('.csv'):
        file_path = os.path.join(normalized_folder, filename)
        df = pd.read_csv(file_path)

        X_state, Y_obs, U_ctrl = preprocess_flight_normalized(df)

        X_list.append(X_state)
        Y_list.append(Y_obs)
        U_list.append(U_ctrl)
        file_names.append(filename)

        #if np.any(np.isnan(X_state)):
        #    print(f"NaN values in {filename}")

        print(f"Processed {filename} -> X:{X_state.shape}, Y:{Y_obs.shape}, U:{U_ctrl.shape}")

# Save combined dataset
#np.savez(combined_output, X=np.array(X_list, dtype=object), Y=np.array(Y_list, dtype=object), U=np.array(U_list, dtype=object))
print(f"\nCombined dataset saved to {combined_output}")


Processed 999_999_Jumper_Anon_ANO_1_20240409-102625_C_OfficialResults_cleaned_quad-cleaned_normalized-quad.csv -> X:(133, 14), Y:(133, 4), U:(133, 20)
Processed 999_999_Jumper_Anon_ANO_1_20240409-102742_C_OfficialResults_cleaned_quad-cleaned_normalized-quad.csv -> X:(129, 14), Y:(129, 4), U:(129, 20)
Processed 999_999_Jumper_Anon_ANO_1_20240409-102831_C_OfficialResults_cleaned_quad-cleaned_normalized-quad.csv -> X:(142, 14), Y:(142, 4), U:(142, 20)
Processed 999_999_Jumper_Anon_ANO_1_20240409-102832_C_OfficialResults_cleaned_quad-cleaned_normalized-quad.csv -> X:(135, 14), Y:(135, 4), U:(135, 20)
Processed 999_999_Jumper_Anon_ANO_1_20240409-102833_C_OfficialResults_cleaned_quad-cleaned_normalized-quad.csv -> X:(132, 14), Y:(132, 4), U:(132, 20)
Processed 999_999_Jumper_Anon_ANO_1_20240409-102834_C_OfficialResults_cleaned_quad-cleaned_normalized-quad.csv -> X:(131, 14), Y:(131, 4), U:(131, 20)
Processed 999_999_Jumper_Anon_ANO_1_20240409-102835_C_OfficialResults_cleaned_quad-cleaned_nor

In [64]:
len(X_list)

204

In [65]:
weird_vrednosti = X_list.pop(169), Y_list.pop(169), U_list.pop(169)
weird_vrednosti

(array([[ 2.68105263e+00, -3.00000000e-02, -3.77631579e-01, ...,
          4.95878947e+00, -1.17835000e+01, -8.96647368e+00],
        [ 4.04342105e+00, -3.00000000e-02, -5.48684211e-01, ...,
          8.23268421e+00, -7.26152632e+00, -3.58710526e+00],
        [ 5.37435897e+00, -3.00000000e-02, -7.43589744e-01, ...,
          1.15616923e+01, -2.73625641e+00,  2.94923077e+00],
        ...,
        [ 1.82308621e+02, -6.22413793e-01, -1.09136897e+02, ...,
          4.71444828e+00, -1.81309655e+01, -1.65136552e+01],
        [ 1.83714138e+02, -5.66551724e-01, -1.10207931e+02, ...,
          3.54920690e+00, -2.03143103e+01, -1.86934138e+01],
        [ 1.85088889e+02, -5.04444444e-01, -1.11277407e+02, ...,
          3.24855556e+00, -2.35774815e+01, -2.34446667e+01]],
       shape=(142, 14)),
 array([[ 2.68105263e+00, -3.00000000e-02, -3.77631579e-01,
          2.89868421e+00],
        [ 4.04342105e+00, -3.00000000e-02, -5.48684211e-01,
          2.94421053e+00],
        [ 5.37435897e+00, -3.00

In [66]:
len(X_list)

203

In [67]:
import numpy as np
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge

def fit_ssm_cv_quad(X_list, U_list, Y_list, n_splits=5, alpha=1e-3):
    """
    Fit an SSM model using K-fold cross-validation across jumps.

    Args:
        X_list, U_list, Y_list: lists of arrays (one per jump)
        n_splits (int): number of folds (or = len(X_list) for LOOCV)
        alpha (float): ridge regularization parameter

    Returns:
        avg_train_err, avg_test_err, models (list of (A,B,C,D) per fold)
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    train_errors, test_errors = [], []
    models = []

    for fold, (train_idx, test_idx) in enumerate(kf.split(X_list)):
        X_t = np.vstack([seq[:-1] for i, seq in enumerate(X_list) if i in train_idx])
        X_next = np.vstack([seq[1:] for i, seq in enumerate(X_list) if i in train_idx])
        U_t = np.vstack([u[:-1] for i, u in enumerate(U_list) if i in train_idx])
        Y_t = np.vstack([y[:-1] for i, y in enumerate(Y_list) if i in train_idx])

        # Regression [X_next] ~ [X_t | U_t]
        Phi = np.hstack([X_t, U_t])
        ridge = Ridge(alpha=alpha, fit_intercept=False)     #ridge is just MNK z regularizacijo
        ridge.fit(Phi, X_next)
        Theta = ridge.coef_.T

        state_dim = X_t.shape[1]
        #control_dim = U_t.shape[1]
        A = Theta[:state_dim, :].T
        B = Theta[state_dim:, :].T

        # Regression [Y_t] ~ [X_t | U_t]
        Phi_y = np.hstack([X_t, U_t])
        ridge_y = Ridge(alpha=alpha, fit_intercept=False)
        ridge_y.fit(Phi_y, Y_t)
        Theta_y = ridge_y.coef_.T

        obs_dim = Y_t.shape[1]
        C = Theta_y[:state_dim, :].T
        D = Theta_y[state_dim:, :].T

        models.append((A, B, C, D))

        train_errs = []
        for i in train_idx:  #should be one, but just in case
            x0 = X_list[i][0]
            U_seq = U_list[i]
            X_seq = X_list[i]
            
            X_sim = [x0]
            for t in range(len(U_seq)-1):
                x_next = A @ X_sim[-1] + B @ U_seq[t]
                X_sim.append(x_next)
            X_sim = np.array(X_sim)
        
            #train_err = error_fun(X_seq[:, :3], X_sim[:, :3])
            train_err = error_fun_x_aligned(X_seq[:, :3], X_sim[:, :3])
            train_errs.append(train_err)

        train_errors.append(np.mean(train_errs))

        test_errs = []
        for j in test_idx:
            x0 = X_list[j][0]      
            U_seq = U_list[j]
            X_seq = X_list[j]

            # simulate with learned model
            X_sim = [x0]
            for t in range(len(U_seq)-1):
                x_next = A @ X_sim[-1] + B @ U_seq[t]
                X_sim.append(x_next)
            X_sim = np.array(X_sim)
            
            #test_err = error_fun(X_seq[:, :3], X_sim[:, :3])
            test_err = error_fun_x_aligned(X_seq[:, :3], X_sim[:, :3])
            test_errs.append(test_err)

            plot(X_seq[:, :3], X_sim[:, :3], j, "15-S3-new_function")
        test_errors.append(np.mean(test_errs))

        #print(train_err, np.mean(test_errs))
        print(np.mean(train_errs), np.mean(test_errs))

    return np.mean(train_errors), np.mean(test_errors), models

In [68]:
avg_train_err, avg_test_err, models = fit_ssm_cv_quad(X_list, U_list, Y_list, n_splits=len(X_list), alpha=10)

print("average train error:", avg_train_err)
print("average test error:", avg_test_err)

1.319485139822634 0.8627632424967214
1.318817503876264 0.7336381513459543
1.3213733915094812 0.4775743689166718
1.3176380517065622 1.067586858499771
1.3189241515772314 0.7624020404351411
1.3100031890784443 2.0086901307447826
1.3162529536010827 0.9150158535582515
1.310180545993016 2.0652873801811698
1.3176244475256793 1.1378807821988666
1.31978547768686 1.0703854409766844
1.318717671917763 0.9844178924046852
1.307920511438181 2.875291589762467
1.3160106109700993 1.5476477123213976
1.3184695958604373 1.6471426953205328
1.3140518077869923 2.0761256869069458
1.3135245750427256 1.0745715906689348
1.3180031177585434 1.1220896314890112
1.3135435179284338 2.0082777637404488
1.3177803054327404 2.1099763129319014
1.3126691054253643 2.679106444394741
1.3159021718578203 1.3092460746061947
1.3175261803249563 0.8773373293663053
1.3192024633759825 2.0511265719771625
1.3187853091653787 0.8048170928655844
1.3149194764904923 2.188146012850266
1.3137630312178283 2.1417755767638744
1.3200621153459282 0.55

# Shranjevanje

In [69]:

def build_models(X_list, U_list, Y_list, n_splits=5, alpha=1e-3):
    """
    Build the matrices and arrays for simulation

    Args:
        X_list, U_list, Y_list: lists of arrays (one per jump)
        n_splits (int): number of folds (or = len(X_list) for LOOCV)
        alpha (float): ridge regularization parameter

    Returns:
        the calculated matrices
    """
    models = []

    X_t = np.vstack([seq[:-1] for i, seq in enumerate(X_list)])
    X_next = np.vstack([seq[1:] for i, seq in enumerate(X_list)])
    U_t = np.vstack([u[:-1] for i, u in enumerate(U_list)])
    Y_t = np.vstack([y[:-1] for i, y in enumerate(Y_list)])

    # Regression [X_next] ~ [X_t | U_t]
    Phi = np.hstack([X_t, U_t])
    ridge = Ridge(alpha=alpha, fit_intercept=False)     #ridge is just MNK z regularizacijo
    ridge.fit(Phi, X_next)
    Theta = ridge.coef_.T

    state_dim = X_t.shape[1]
    control_dim = U_t.shape[1]
    A = Theta[:state_dim, :].T
    B = Theta[state_dim:, :].T

        # Regression [Y_t] ~ [X_t | U_t]
    Phi_y = np.hstack([X_t, U_t])
    ridge_y = Ridge(alpha=alpha, fit_intercept=False)
    ridge_y.fit(Phi_y, Y_t)
    Theta_y = ridge_y.coef_.T

    obs_dim = Y_t.shape[1]
    C = Theta_y[:state_dim, :].T
    D = Theta_y[state_dim:, :].T

    models = [A, B, C, D]

    return models

In [70]:
models = build_models(X_list, U_list, Y_list)

In [71]:
def app_sim_AVGbase(X_important, models, sliders=np.zeros(20)):

    A, B = models[0], models[1]
    winds = sliders[0:12]
    angles = sliders[12:20]
    x0 = X_important[0]
    X_sim = [x0]
    X_coord = [x0[:3]]
    U_seq = []
    file_length = np.shape(X_important)[0]

    winds_quad = np.square(winds)

    winds_angles = np.concatenate([winds, winds_quad, angles])

    for i in range(file_length):
        U_seq.append(winds_angles)

    for t in range(file_length-1):

        x_next = A @ X_sim[-1] + B @ U_seq[t]
        X_sim.append(x_next)
        X_coord.append(x_next[:3])
    X_sim = np.array(X_sim)

    return X_coord

In [72]:
state_names = [
    "X [m]", "Y [m]", "Z [m]",
    "Vx [m/s]", "Vy [m/s]", "Vz [m/s]",
    "Speed [m/s]",
    "Opening Angle [°]",
    "Roll Angle Left [°]", "Roll Angle Right [°]",
    "Yaw Angle Left [°]", "Yaw Angle Right [°]",
    "Stalling Angle Left [°]", "Stalling Angle Right [°]"]

In [73]:
control_names = []

for feature in wind_features:
    for zone in zones.keys():
        control_names.append(f"{zone}_{feature}")

angle_control_names = [
    "Speed [m/s]",
    "Opening Angle [°]",
    "Roll Angle Left [°]",
    "Roll Angle Right [°]",
    "Yaw Angle Left [°]",
    "Yaw Angle Right [°]",
    "Stalling Angle Left [°]",
    "Stalling Angle Right [°]" ]

control_names.extend(angle_control_names)

In [74]:
control_names

['takeoff_Speed',
 'mid_Speed',
 'landing_Speed',
 'takeoff_Tangent',
 'mid_Tangent',
 'landing_Tangent',
 'takeoff_Cross',
 'mid_Cross',
 'landing_Cross',
 'takeoff_Turbulence',
 'mid_Turbulence',
 'landing_Turbulence',
 'Speed [m/s]',
 'Opening Angle [°]',
 'Roll Angle Left [°]',
 'Roll Angle Right [°]',
 'Yaw Angle Left [°]',
 'Yaw Angle Right [°]',
 'Stalling Angle Left [°]',
 'Stalling Angle Right [°]']

In [75]:
A, B, C, D = models[0], models[1], models[2], models[3]

np.savetxt("matrixA2.csv", A, delimiter=",")
np.savetxt("matrixB2.csv", B, delimiter=",")
np.savetxt("matrixC2.csv", C, delimiter=",")
np.savetxt("matrixD2.csv", D, delimiter=",")

In [76]:
A_df = pd.DataFrame(A, index=state_names, columns=state_names)
A_df.to_csv("matrixA_named2.csv")

B_df = pd.DataFrame(B, index=state_names, columns=control_names)
B_df.to_csv("matrixB_named2.csv")

## with avg base

In [77]:
filename = "average_flight_normalized.csv"
df = pd.read_csv(filename)

Xa_state, Ya_obs, Ua_ctrl = preprocess_flight_normalized(df)

print(f"Processed {filename} -> X:{Xa_state.shape}, Y:{Ya_obs.shape}, U:{Ua_ctrl.shape}")

Processed average_flight_normalized.csv -> X:(139, 14), Y:(139, 4), U:(139, 20)


In [78]:
import numpy as np
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge

def fit_ssm_cv_AVG(X_list, U_list, Y_list, n_splits=5, alpha=1e-3):
    """
    Fit an SSM model using K-fold cross-validation across jumps.

    Args:
        X_list, U_list, Y_list: lists of arrays (one per jump)
        n_splits (int): number of folds (or = len(X_list) for LOOCV)
        alpha (float): ridge regularization parameter

    Returns:
        avg_train_err, avg_test_err, models (list of (A,B,C,D) per fold)
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    train_errors, test_errors = [], []
    models = []

    for fold, (train_idx, test_idx) in enumerate(kf.split(X_list)):
        X_t = np.vstack([seq[:-1] for i, seq in enumerate(X_list) if i in train_idx])
        X_next = np.vstack([seq[1:] for i, seq in enumerate(X_list) if i in train_idx])
        U_t = np.vstack([u[:-1] for i, u in enumerate(U_list) if i in train_idx])
        Y_t = np.vstack([y[:-1] for i, y in enumerate(Y_list) if i in train_idx])

        # Regression [X_next] ~ [X_t | U_t]
        Phi = np.hstack([X_t, U_t])
        ridge = Ridge(alpha=alpha, fit_intercept=False)     #ridge is just MNK z regularizacijo
        ridge.fit(Phi, X_next)
        Theta = ridge.coef_.T

        state_dim = X_t.shape[1]
        #control_dim = U_t.shape[1]
        A = Theta[:state_dim, :].T
        B = Theta[state_dim:, :].T

        # Regression [Y_t] ~ [X_t | U_t]
        Phi_y = np.hstack([X_t, U_t])
        ridge_y = Ridge(alpha=alpha, fit_intercept=False)
        ridge_y.fit(Phi_y, Y_t)
        Theta_y = ridge_y.coef_.T

        obs_dim = Y_t.shape[1]
        C = Theta_y[:state_dim, :].T
        D = Theta_y[state_dim:, :].T

        models.append((A, B, C, D))

        train_errs = []
        for i in train_idx:  #should be one, but just in case
            x0 = Xa_state[0]
            U_seq = U_list[i]
            X_seq = X_list[i]
            
            X_sim = [x0]
            for t in range(len(U_seq)-1):
                x_next = A @ X_sim[-1] + B @ U_seq[t]
                X_sim.append(x_next)
            X_sim = np.array(X_sim)
        
            #train_err = error_fun(X_seq[:, :3], X_sim[:, :3])
            train_err = error_fun_x_aligned(X_seq[:, :3], X_sim[:, :3])
            #print(f"X_seq(x, y, z) = {X_seq[:, :3]}")
            #print(f"X_sim(x, y, z) = {X_sim[:, :3]}")
            #print(train_err)
            train_errs.append(train_err)

        train_errors.append(np.mean(train_errs))

        test_errs = []
        for j in test_idx:
            x0 = Xa_state[0]      
            U_seq = U_list[j]
            X_seq = X_list[j]

            # simulate with learned model
            X_sim = [x0]
            for t in range(len(U_seq)-1):
                x_next = A @ X_sim[-1] + B @ U_seq[t]
                X_sim.append(x_next)
            X_sim = np.array(X_sim)
            
            #test_err = error_fun(X_seq[:, :3], X_sim[:, :3])
            test_err = error_fun_x_aligned(X_seq[:, :3], X_sim[:, :3])
            test_errs.append(test_err)

            #print("real x0:", X_list[j][0][:3])
            #print("used x0:", x0[:3])
            #print("X_seq x-range:", X_seq[:,0].min(), X_seq[:,0].max(), "len", len(X_seq))
            #print("X_sim x-range:", X_sim[:,0].min(), X_sim[:,0].max(), "len", len(X_sim))
            #
            #c1 = interpolate_curve_by_x(X_seq[:, :3])
            #c2 = interpolate_curve_by_x(X_sim[:, :3])
            #print("interp lens:", len(c1), len(c2))
            #print("interp x-range:", (c1[:,0].min(), c1[:,0].max()), (c2[:,0].min(), c2[:,0].max()))
            #print("first 5 xs:", c1[:5,0], c2[:5,0])


            plot(X_seq[:, :3], X_sim[:, :3], j, "16-S3-new_function")
        test_errors.append(np.mean(test_errs))

        #print(train_err, np.mean(test_errs))
        print(np.mean(train_errs), np.mean(test_errs))


    return np.mean(train_errors), np.mean(test_errors), models

In [79]:
avg_train_err, avg_test_err, models = fit_ssm_cv_AVG(X_list, U_list, Y_list, n_splits=len(X_list), alpha=10)

print("average train error:", avg_train_err)
print("average test error:", avg_test_err)

1.5754636093563392 1.486179942408091
1.577474877482555 1.0430907980676343
1.5792559648368711 1.0603862319413437
1.5753146521808712 1.5243218255004527
1.5760217029284282 1.2254342722175111
1.564327698524972 2.03054091204375
1.5880947193557389 1.3092808583416944
1.5676303267890315 2.1833807382366213
1.5732160754389974 1.4308361162129906
1.5783182182119098 1.0394769031249351
1.5771072929176908 1.7169885042897959
1.5661709138724649 3.206402526710082
1.5759201329169377 1.362981410871028
1.5782217874134168 1.7082069101946484
1.5695091525386018 2.3552244989348496
1.5669067080368806 1.120401685252115
1.5777043106416173 1.260204945594463
1.5838986544123506 1.9071562164085305
1.583638474487259 2.2249590327906863
1.5702236709016069 2.7808391384874023
1.5724836216661433 1.862064604046183
1.576373772722681 1.041177826196745
1.5807638876549852 2.0282982951483066
1.574611138054145 1.0670669293316377
1.5754632992403286 2.2311969328513968
1.5681823692261494 2.7260737601061362
1.5787774726533166 0.80941

In [80]:

file_names = []
Xa_list, Ya_list, Ua_list = [], [], []

filename = "average_flight_normalized.csv"
df = pd.read_csv(filename)

Xa_state, Ya_obs, Ua_ctrl = preprocess_flight_normalized(df)

Xa_list.append(Xa_state)
Ya_list.append(Ya_obs)
Ua_list.append(Ua_ctrl)
file_names.append(filename)

print(f"Processed {filename} -> X:{Xa_state.shape}, Y:{Ya_obs.shape}, U:{Ua_ctrl.shape}")

Processed average_flight_normalized.csv -> X:(139, 14), Y:(139, 4), U:(139, 20)


In [81]:
models = build_models(Xa_list, Ua_list, Ya_list)

In [82]:
sum(sum(np.isnan(Ua_list)))

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [83]:
A, B, C, D = models[0], models[1], models[2], models[3]

np.savetxt("matrixA3.csv", A, delimiter=",")
np.savetxt("matrixB3.csv", B, delimiter=",")
np.savetxt("matrixC3.csv", C, delimiter=",")
np.savetxt("matrixD3.csv", D, delimiter=",")


### baseline

In [84]:
import numpy as np
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge

def baseline(X_list, U_list, Y_list, n_splits=5, alpha=1e-3):
    """
    compares the error with an average flight instead of the actual trajectory

    Args:
        X_list, U_list, Y_list: lists of arrays (one per jump)
        n_splits (int): number of folds (or = len(X_list) for LOOCV)
        alpha (float): ridge regularization parameter

    Returns:
        avg_train_err, avg_test_err, models (list of (A,B,C,D) per fold)
    """

    file_name = "average_flight.csv"
    df = pd.read_csv(file_name, delimiter=',')
    x = df["X [m]"][1:].astype(float).to_numpy()
    y = df["Y [m]"][1:].astype(float).to_numpy()
    z = df["Z [m]"][1:].astype(float).to_numpy()
    X_avg = np.column_stack((x, y, z))

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    train_errors, test_errors = [], []
    models = []

    for fold, (train_idx, test_idx) in enumerate(kf.split(X_list)):
        X_t = np.vstack([seq[:-1] for i, seq in enumerate(X_list) if i in train_idx])
        X_next = np.vstack([seq[1:] for i, seq in enumerate(X_list) if i in train_idx])
        U_t = np.vstack([u[:-1] for i, u in enumerate(U_list) if i in train_idx])
        Y_t = np.vstack([y[:-1] for i, y in enumerate(Y_list) if i in train_idx])

        train_errs = []
        for i in train_idx:  #should be one, but just in case
            x0 = Xa_state[0]
            U_seq = U_list[i]
            X_seq = X_list[i]
        
            train_err = error_fun_x_aligned(X_seq[:, :3], X_avg)
            train_errs.append(train_err)
        train_errors.append(np.mean(train_errs))

        test_errs = []
        for j in test_idx:
            x0 = Xa_state[0]      
            U_seq = U_list[j]
            X_seq = X_list[j]
            
            #test_err = error_fun(X_seq[:, :3], X_sim[:, :3])
            test_err = error_fun_x_aligned(X_seq[:, :3], X_avg)
            test_errs.append(test_err)


            plot(X_seq[:, :3], X_avg, j, "17_avg-baseline")
        test_errors.append(np.mean(test_errs))

        #print(train_err, np.mean(test_errs))
        print(np.mean(train_errs), np.mean(test_errs))


    return np.mean(train_errors), np.mean(test_errors), models

In [85]:
avg_train_err, avg_test_err, models = baseline(X_list, U_list, Y_list, n_splits=len(X_list), alpha=10)

print("average train error:", avg_train_err)
print("average test error:", avg_test_err)

2.7264619637301086 1.5845255915872114
2.7314648318491215 0.5739462315466758
2.6877128752408668 9.4118414664142
2.7293663657839047 0.9978363767205185
2.7225925150037726 2.366154234307071
2.7082504266717384 5.263256077377941
2.721147906958966 2.6579650593581277
2.717595389179563 3.3755736507973704
2.715299269030037 3.839389921001702
2.730537364834711 0.7612945684575549
2.7296283809908557 0.944909304916295
2.7194010344761237 3.0108333008922132
2.7149942757236896 3.900998568883953
2.72751049047621 1.372723188874799
2.7194915952853966 2.992540017419119
2.716136257676158 3.670318214485318
2.729604675508774 0.9496978122967997
2.724772066477992 1.9258848365147574
2.724510608684614 1.9786993107771982
2.723320836039992 2.2190333849908126
2.7136476944208443 4.173007992058676
2.7227373815435447 2.3368911932730887
2.7213341324287366 2.6203475144644037
2.7215904401858477 2.568573347527972
2.7190864770608885 3.074373898769575
2.698565690787991 7.219572725895035
2.729243042411798 1.0227476978860461
2.